In [2]:
from odc.geo.gridspec import GridSpec
from odc.dscache.tools.tiling import tile_shape_standard
from odc.geo import CRS, yx_, Geometry

import geopandas as gpd

In [3]:
# Use the DEA example as a starting point
# https://github.com/opendatacube/odc-dscache/blob/2dc288b379945627ba2b1c58b1fa175bbaf2189b/odc/dscache/tools/tiling.py#L51C4-L71C7

# This doesn't actually do anything significant, because the
# transform from both GDA94 and GDA2020 to WGS84, which happens
# on conversion to geojson, is effectively null.
albers_gda2020 = CRS("EPSG:9473")

gridspec = GridSpec(
    crs=albers_gda2020,
    tile_shape=tile_shape_standard,
    resolution=96_000,
    origin=yx_(-6912000.0, -4416000.0)
)

In [ ]:
geoboundaries_au = (
    "https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/"
    "bdfb316b1fdcac1473051979152cac0943c549fa/releaseData/gbOpen/AUS/ADM0/geoBoundaries-AUS-ADM0-all.zip"
)
boundaries_au, layer="geoBoundaries-AUS-ADM1-all — geoBoundaries-AUS-ADM1.shp")

protected_areas = (
    "https://hub.arcgis.com/api/v3/datasets/ec356a872d8048459fe78fc80213dc70_0/"
    "downloads/data?format=shp&spatialRefId=4283&where=1%3D1"
)

protected_areas_gdf = gpd.read_file(protected_areas).to_crs("epsg:4326")

/Users/alex/git/auspatious/grid-extent-example/.venv/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
/Users/alex/git/auspatious/grid-extent-example/.venv/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: Geometry of polygon of fid 5897 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


In [ ]:
australia.to_file("temp_australia.shp.zip", driver="ESRI Shapefile")

In [ ]:
# Keep protected areas that don't intersect the Natural Earth Australia geometry.
australia_geometry = australia.geometry.union_all()
protected_areas_not_in_aus = protected_areas_gdf[~protected_areas_gdf.intersects(australia_geometry)]
print(f"Limited protected areas to {len(protected_areas_not_in_aus)} out of {len(protected_areas_gdf)}")

In [ ]:
# Buffer by 50 km in a projected CRS
buffered_australia_extent = gpd.GeoSeries(
    [australia_geometry, *protected_areas_not_in_aus.geometry],
    crs=australia.crs,
).to_crs(albers_gda2020).buffer(100_000)

# Dissolve overlapping buffered polygons into one geometry.
australia_extent = gpd.GeoSeries([buffered_australia_extent.union_all()], crs=albers_gda2020)
australia_extent.explore()

In [ ]:
dc_geom = Geometry(australia_extent.geometry.union_all(), crs=albers_gda2020)
geometry = gridspec.geojson(geopolygon=dc_geom)
# GridSpec.geojson() emits WGS84/CRS84 coordinates.
tiles = gpd.GeoDataFrame.from_features(geometry, crs="EPSG:4326")
tiles.to_file("tiles.geojson", driver="GeoJSON")

In [ ]:
from pyproj import network
from pyproj.transformer import TransformerGroup
from shapely.ops import transform as transform_geometry

network.set_network_enabled(True)

# Here we explicitly label the data GDA94, which triggers the real
# changing of coordinates...
tiles_gda94 = tiles.to_crs("EPSG:4283")

transformers = TransformerGroup(
    tiles_gda94.crs,
    albers_gda2020,
    always_xy=True,
    allow_ballpark=False,
)
known_accuracy = [
    transformer for transformer in transformers.transformers if transformer.accuracy >= 0
]
if not known_accuracy:
    raise RuntimeError("PROJ did not provide a known-accuracy transformation.")

# This is a high-accuracy transformation, using the grid shift files provided by PROJ.
# resulting in an offset if ~1.7 m.
transformer = min(known_accuracy, key=lambda candidate: candidate.accuracy)
print(f"Using: {transformer.description}")
print(f"Accuracy: {transformer.accuracy} m")

# Transform the WGS84 tile geometries into Australian Albers GDA2020.
tiles_albers = tiles_gda94.copy()
tiles_albers["geometry"] = tiles_gda94.geometry.map(
    lambda geometry: transform_geometry(transformer.transform, geometry)
)
tiles_albers = tiles_albers.set_crs(albers_gda2020, allow_override=True)
tiles_albers.to_file("tiles_albers.shp.zip", driver="ESRI Shapefile")
tiles_albers.to_crs("epsg:4326").to_file("tiles.geojson", driver="GeoJSON")